# Seminar 13: Supervised Fine-Tuning for Polite Rewriting

Today is a small instruction-tuning lab.

Goals:
- format instruction examples into prompt-response text,
- probe a base causal LM before supervised fine-tuning,
- understand response-only label masking,
- assemble a lightweight SFT pipeline,
- compare generations before and after SFT.

Expected rhythm: 5 medium board tasks, about 90 minutes total.


## 0. Setup

If you run this in Colab and packages are missing, uncomment the install line first.


In [ ]:
# If needed in Colab, uncomment:
# %pip install transformers datasets accelerate -q

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, default_data_collator

pd.set_option('display.max_colwidth', 180)


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


### Provided Dataset Upload, Split, and Model Loading

This is setup, not a board exercise.

Upload one JSONL file with examples containing three fields:
- `instruction`: what the model should do,
- `input`: the impolite/direct message,
- `response`: the desired polite rewrite.

The notebook will split this single file into train and validation sets.

In Colab, the cell below opens an upload dialog. Outside Colab, it uses a local fallback file if available.

We use `distilgpt2` because it is small enough for a classroom SFT demo.


In [ ]:
from pathlib import Path
import json
import random
from datasets import Dataset, DatasetDict

CFG = {
    'model_name': 'distilgpt2',
    'max_length': 128,
    'train_batch_size': 4,
    'eval_batch_size': 4,
    'learning_rate': 5e-5,
    'num_epochs': 2,
    'seed': 42,
    'max_new_tokens': 48,
    'validation_fraction': 0.15,
}


def find_local_dataset_file():
    candidate_paths = [
        Path('polite_rewriting_dataset.jsonl'),
        Path('/content/polite_rewriting_dataset.jsonl'),
    ]
    for path in candidate_paths:
        if path.exists():
            return path
    return None


def choose_dataset_file():
    try:
        from google.colab import files
        print('Please upload one JSONL dataset file.')
        uploaded = files.upload()
        uploaded_names = list(uploaded.keys())
        if len(uploaded_names) == 0:
            raise ValueError('No file was uploaded.')
        return Path(uploaded_names[0])
    except ModuleNotFoundError:
        local_file = find_local_dataset_file()
        if local_file is None:
            raise FileNotFoundError('Place polite_rewriting_dataset.jsonl in the notebook folder, or run this notebook in Colab and upload it.')
        print('Using local dataset file:', local_file)
        return local_file


def read_jsonl_dataset(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if line == '':
                continue
            rows.append(json.loads(line))
    return rows


def validate_rows(rows):
    required_fields = ['instruction', 'input', 'response']
    for row_index in range(len(rows)):
        row = rows[row_index]
        for field in required_fields:
            if field not in row:
                raise ValueError(f'Row {row_index} is missing field: {field}')
            if not isinstance(row[field], str):
                raise ValueError(f'Row {row_index} field {field} must be a string.')


def split_rows(rows, validation_fraction, seed):
    shuffled_rows = rows.copy()
    rng = random.Random(seed)
    rng.shuffle(shuffled_rows)

    val_size = max(1, int(len(shuffled_rows) * validation_fraction))
    train_rows = shuffled_rows[val_size:]
    val_rows = shuffled_rows[:val_size]
    return train_rows, val_rows


dataset_file = choose_dataset_file()
all_rows = read_jsonl_dataset(dataset_file)
validate_rows(all_rows)
train_rows, val_rows = split_rows(all_rows, CFG['validation_fraction'], CFG['seed'])

dataset = DatasetDict({
    'train': Dataset.from_list(train_rows),
    'validation': Dataset.from_list(val_rows),
})

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(CFG['model_name']).to(device)
model.config.pad_token_id = tokenizer.pad_token_id

print(dataset)
print('dataset file:', dataset_file)
print('all examples:', len(all_rows))
print('train examples:', len(dataset['train']))
print('validation examples:', len(dataset['validation']))

preview_rows = []
preview_count = min(3, len(dataset['train']))
for i in range(preview_count):
    example = dataset['train'][i]
    preview_rows.append({
        'instruction': example['instruction'],
        'input': example['input'],
        'response': example['response'],
    })

pd.DataFrame(preview_rows)


## 1. Exercise 1: Prompt Format Lab

SFT teaches a model to respond to a particular interface. The prompt format matters.

Task:
- implement `build_prompt`,
- implement `build_full_text`,
- inspect one formatted training example,
- create one intentionally changed prompt format for discussion.

Expected implementation:
1. In `build_prompt`, concatenate three labeled sections: `### Instruction:`, `### Input:`, and `### Response:`.
2. Put the example instruction after the Instruction header.
3. Put the example input message after the Input header.
4. End the prompt immediately after `### Response:
`; do not include the target response there.
5. In `build_full_text`, append the target response and `tokenizer.eos_token`.
6. Print both strings and check that the prompt is what you would use at inference time, while full text is what you train on.

Function contracts:
- `build_prompt(example)` returns the text up to `### Response:
`, without the target response,
- `build_full_text(example, tokenizer)` returns prompt + target response + EOS token,
- the full text should contain Instruction, Input, and Response sections.

Useful note:
The same prompt format should be used during training and inference.


In [ ]:
def build_prompt(example):
    # 1. Start with the Instruction section.
    # 2. Add the example instruction text.
    # 3. Add the Input section and the original user message.
    # 4. Add the Response section header, but not the response itself.
    return ''


def build_full_text(example, tokenizer):
    # 1. Reuse build_prompt(example).
    # 2. Add the target polite response.
    # 3. Add tokenizer.eos_token so the model learns where the answer ends.
    return ''


example = dataset['train'][0]
formatted_prompt = build_prompt(example)
formatted_full_text = build_full_text(example, tokenizer)

print(formatted_prompt)
print('--- full training text ---')
print(formatted_full_text)


### Checks (Exercise 1)

In [ ]:
assert isinstance(formatted_prompt, str)
assert isinstance(formatted_full_text, str)
for section_name in ['### Instruction:', '### Input:', '### Response:']:
    assert section_name in formatted_prompt
    assert section_name in formatted_full_text

assert example['response'].strip() not in formatted_prompt
assert example['response'].strip() in formatted_full_text
assert formatted_full_text.endswith(tokenizer.eos_token)

print('Exercise 1 passed.')


## 2. Exercise 2: Base Model Behavior Probe

Before SFT, the base model can continue text, but it has not learned this polite-rewriting task.

Task:
- implement `make_generation_prompt`,
- implement `generate_rewrite`,
- run several prompts before fine-tuning,
- build a table of base-model outputs.

Expected implementation:
1. `make_generation_prompt` should create an example-like dictionary with a fixed instruction and the user text as `input`.
2. Reuse `build_prompt` so training and inference formats match.
3. In `generate_rewrite`, tokenize the prompt with `return_tensors='pt'` and move it to `device`.
4. Call `model.generate` with `max_new_tokens`, `do_sample=True`, `temperature`, `pad_token_id`, and `eos_token_id`.
5. Decode the full generated text.
6. Remove the original prompt prefix so the function returns only the generated rewrite.
7. `build_generation_table` should loop over prompts and store one row per input.

Function contracts:
- `make_generation_prompt(user_text)` returns the inference-time prompt ending with `### Response:
`,
- `generate_rewrite(user_text, model, tokenizer)` returns only the generated response part,
- `build_generation_table(prompts, model, tokenizer, column_name)` returns columns `input` and `column_name`.

Useful generation arguments:
- `max_new_tokens` controls response length,
- `do_sample=True` makes generation less deterministic,
- `temperature` controls randomness.


In [ ]:
test_prompts = [
    'Send me the report today.',
    'Why did you ignore my email?',
    'Fix the formatting before sharing this.',
    'Tell me if you are coming to the meeting.',
]


def make_generation_prompt(user_text):
    # Create the same prompt format as training, but with an empty response.
    # The fixed instruction should ask for a polite professional rewrite.
    return ''


def generate_rewrite(user_text, model, tokenizer, temperature=0.7):
    # 1. Build the generation prompt.
    # 2. Tokenize it and move tensors to device.
    # 3. Use model.generate to produce new tokens.
    # 4. Decode the full text.
    # 5. Remove the prompt prefix and return only the generated response.
    return ''


def build_generation_table(prompts, model, tokenizer, column_name):
    rows = []
    for prompt in prompts:
        generated = generate_rewrite(prompt, model, tokenizer)
        rows.append({
            'input': prompt,
            column_name: generated,
        })
    return pd.DataFrame(rows)



before_sft_table = build_generation_table(test_prompts, model, tokenizer, 'before_sft')
before_sft_table


### Checks (Exercise 2)

In [ ]:
assert isinstance(before_sft_table, pd.DataFrame)
for column in ['input', 'before_sft']:
    assert column in before_sft_table.columns
assert len(before_sft_table) == len(test_prompts)

for output in before_sft_table['before_sft']:
    assert isinstance(output, str)

print('Exercise 2 passed.')


## 3. Exercise 3: Response-Only Label Masking

The model receives prompt + response as one sequence. But we usually want the loss to train the response, not force the model to memorize the prompt.

Key idea:
```text
input_ids: prompt tokens + response tokens
labels:    -100 for prompt tokens, real token ids for response tokens
```

PyTorch ignores label value `-100` in the loss.

Task:
- implement `preprocess_example`,
- implement `build_label_mask_table`,
- inspect which tokens contribute to the loss.

Expected implementation:
1. Build `prompt` and `full_text` using the functions from Exercise 1.
2. Tokenize the prompt alone to know how many initial tokens belong to the prompt.
3. Tokenize the full text to get the actual model input sequence.
4. Truncate `full_ids` to `CFG['max_length']`.
5. Start `labels` as a copy of `full_ids`.
6. Replace prompt-token labels with `-100`.
7. Pad `input_ids`, `attention_mask`, and `labels` to `CFG['max_length']`.
8. In `build_label_mask_table`, show token-by-token whether each position contributes to the loss.

Function contracts:
- `preprocess_example(example)` returns `input_ids`, `attention_mask`, `labels`, each of length `CFG['max_length']`,
- prompt label positions should be `-100`,
- at least one response token should contribute to loss,
- `build_label_mask_table(processed, tokenizer)` returns token-level rows for inspection.


In [ ]:
def preprocess_example(example):
    # 1. Build prompt and full training text.
    # 2. Tokenize prompt alone to find prompt length.
    # 3. Tokenize full text to get input_ids.
    # 4. Copy input_ids into labels.
    # 5. Set labels for prompt positions to -100.
    # 6. Pad input_ids, attention_mask, and labels to CFG['max_length'].
    return {
        'input_ids': [],
        'attention_mask': [],
        'labels': [],
    }


def build_label_mask_table(processed, tokenizer):
    tokens = tokenizer.convert_ids_to_tokens(processed['input_ids'])
    rows = []

    for position in range(len(tokens)):
        label = processed['labels'][position]
        rows.append({
            'position': position,
            'token': tokens[position],
            'input_id': processed['input_ids'][position],
            'label': label,
            'contributes_to_loss': label != -100,
        })
    return pd.DataFrame(rows)


processed_example = preprocess_example(dataset['train'][0])
label_mask_table = build_label_mask_table(processed_example, tokenizer)
label_mask_table.head(40)


### Checks (Exercise 3)

In [ ]:
for key in ['input_ids', 'attention_mask', 'labels']:
    assert key in processed_example
    assert len(processed_example[key]) == CFG['max_length']

assert isinstance(label_mask_table, pd.DataFrame)
for column in ['position', 'token', 'input_id', 'label', 'contributes_to_loss']:
    assert column in label_mask_table.columns

assert label_mask_table['contributes_to_loss'].sum() > 0
assert (label_mask_table['label'] == -100).sum() > 0

print('Exercise 3 passed.')


## 4. Exercise 4: Training Pipeline Assembly

Now turn the preprocessing function into a training pipeline.

Task:
- map `preprocess_example` over train and validation splits,
- create `TrainingArguments`,
- create the `Trainer`.

Expected implementation:
1. Use `dataset['train'].map(preprocess_example)` to build `tokenized_train`.
2. Use `dataset['validation'].map(preprocess_example)` to build `tokenized_val`.
3. Create `TrainingArguments` with the provided output directory, batch sizes, learning rate, epochs, logging, and no model saving.
4. Create `Trainer` with the model, training args, tokenized datasets, `default_data_collator`, and tokenizer/processing class.
5. Do not call `trainer.train()` here; training is in the demo cell below.

Useful notes:
- examples are already padded to `CFG['max_length']`, so `default_data_collator` can stack them,
- `Trainer` receives model, args, datasets, and data collator,
- training itself is a demo cell below because waiting is not the exercise.


In [ ]:
# Map preprocessing over both dataset splits.
tokenized_train = None
tokenized_val = None

# Create TrainingArguments with the values in CFG.
training_args = TrainingArguments(
    output_dir='distilgpt2_polite_rewrite',
    eval_strategy='epoch',
    save_strategy='no',
    learning_rate=CFG['learning_rate'],
    per_device_train_batch_size=CFG['train_batch_size'],
    per_device_eval_batch_size=CFG['eval_batch_size'],
    num_train_epochs=CFG['num_epochs'],
    logging_steps=10,
    report_to='none',
    fp16=torch.cuda.is_available(),
    seed=CFG['seed'],
)

# Create Trainer with model, args, datasets, default_data_collator, and tokenizer/processing class.
trainer = None

print('Training pipeline prepared.')


### Checks (Exercise 4)

In [ ]:
assert tokenized_train is not None
assert tokenized_val is not None

first_item = tokenized_train[0]
for key in ['input_ids', 'attention_mask', 'labels']:
    assert key in first_item
    assert len(first_item[key]) == CFG['max_length']

assert training_args is not None
assert trainer is not None

print('Exercise 4 passed.')


## 5. Fine-Tuning Demo

Training is the waiting part. Run this after the pipeline is ready.


In [ ]:
# Run this cell when the pipeline is ready.
train_result = trainer.train()
validation_metrics = trainer.evaluate()
validation_metrics

## 6. Exercise 5: Before vs After SFT Comparison

Now rerun the same prompts and compare behavior.

Task:
- generate after-SFT outputs for the same prompts,
- add held-out prompts,
- build a comparison table,
- write short notes on what improved and what still failed.

Expected implementation:
1. Combine the original `test_prompts` with the new `heldout_prompts`.
2. Build a lookup from `before_sft_table` so old prompts can reuse their before-SFT outputs.
3. For each prompt, call `generate_rewrite` with the fine-tuned model.
4. Store `input`, `before_sft`, `after_sft`, and a short `note` in the table.
5. For held-out prompts, `before_sft` may be empty because they were not generated before training.
6. Fill `comparison_notes` with concrete observations from the table, not generic claims.

Function contract:
- `build_before_after_table(prompts, before_table, model, tokenizer)` returns columns `input`, `before_sft`, `after_sft`, `note`.

Good notes to discuss:
- Did the model become more polite?
- Did it preserve the original intent?
- Did it copy the prompt format or produce extra text?
- Did it generalize to held-out prompts?


In [ ]:
heldout_prompts = [
    'Your slides are messy and hard to read.',
    'You forgot to send the invoice again.',
    'This answer is wrong. Redo it.',
]

all_comparison_prompts = test_prompts + heldout_prompts


def build_before_after_table(prompts, before_table, model, tokenizer):
    rows = []
    # 1. Create a lookup from before_table: input text -> before_sft output.
    # 2. Loop over prompts.
    # 3. Generate after_sft output with the fine-tuned model.
    # 4. Append a row with input, before_sft, after_sft, and note.
    return pd.DataFrame(rows)


comparison_table = build_before_after_table(all_comparison_prompts, before_sft_table, model, tokenizer)

comparison_table


### Checks (Exercise 5)

In [ ]:
assert isinstance(comparison_table, pd.DataFrame)
for column in ['input', 'before_sft', 'after_sft', 'note']:
    assert column in comparison_table.columns

assert len(comparison_table) == len(all_comparison_prompts)
for output in comparison_table['after_sft']:
    assert isinstance(output, str)

print('Exercise 5 passed.')


## 7. Wrap-Up Questions

1. What did SFT visibly change in the model behavior?
2. What did SFT not teach the model?
3. Why do we mask prompt tokens with `-100`?
4. Why should training and inference prompt formats match?
5. Did the model generalize to held-out prompts?
6. How is this different from pretraining?


## Optional Extension

Try one small experiment:
- remove `### Response:` at inference time,
- change the instruction wording,
- train for fewer examples,
- train for more epochs.

Question: did the model learn the general polite-rewriting behavior, or mostly the exact classroom interface?
